# Derive Andvari Network Enforcement Summary

This notebook derives a compact proxy/firewall analysis artifact from `andvari_invocations.csv`.
The derived CSV is intended for the small thesis results subsection on proxy and firewall enforcement events.

It produces:

- `andvari_network_enforcement_summary.csv`, with one row per Andvari invocation.
- A compact table grouped by agent and reconstruction variant.
- A LaTeX table matching the thesis result presentation.


In [ ]:
from pathlib import Path

import pandas as pd


def find_package_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'data' / 'exported' / 'andvari_invocations.csv').exists():
            return path
    raise FileNotFoundError('Could not find data/exported/andvari_invocations.csv')


PACKAGE_ROOT = find_package_root(Path.cwd())
INPUT_CSV = PACKAGE_ROOT / 'data' / 'exported' / 'andvari_invocations.csv'
OUTPUT_CSV = PACKAGE_ROOT / 'data' / 'derived' / 'andvari_network_enforcement_summary.csv'

df = pd.read_csv(INPUT_CSV)
df.head()


: 

## Validate required columns

The analysis only uses columns that already exist in `andvari_invocations.csv`. The derived fields are computed from the proxy and firewall counts.

In [ ]:
required = [
    'agent',
    'blocked_egress_line_count',
    'proxy_access_line_count',
    'proxy_denied_count',
    'reason',
    'repo_slug',
    'run_id',
    'status',
    'step',
    'variant',
]

missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

df[required].head()


## Derive enforcement fields

An invocation is counted as denied or blocked when either:

- `proxy_denied_count > 0`, or
- `blocked_egress_line_count > 0`.


In [ ]:
summary = df[required].copy()

for col in ['proxy_access_line_count', 'proxy_denied_count', 'blocked_egress_line_count']:
    summary[col] = pd.to_numeric(summary[col], errors='coerce').fillna(0).astype(int)

summary['denied_or_blocked'] = (
    (summary['proxy_denied_count'] > 0) |
    (summary['blocked_egress_line_count'] > 0)
)

def classify_event_type(row):
    proxy = row['proxy_denied_count'] > 0
    firewall = row['blocked_egress_line_count'] > 0
    if proxy and firewall:
        return 'proxy_denial_and_firewall_block'
    if proxy:
        return 'proxy_denial'
    if firewall:
        return 'firewall_block'
    return 'none'

summary['main_event_type'] = summary.apply(classify_event_type, axis=1)

variant_labels = {
    'generated': 'Best-effort',
    'v2': 'Slightly degraded',
    'v3': 'Moderately degraded',
}
summary['variant_label'] = summary['variant'].map(variant_labels).fillna(summary['variant'])

summary = summary[
    [
        'run_id',
        'repo_slug',
        'agent',
        'variant',
        'variant_label',
        'step',
        'status',
        'reason',
        'proxy_access_line_count',
        'proxy_denied_count',
        'blocked_egress_line_count',
        'denied_or_blocked',
        'main_event_type',
    ]
]

summary.head()


## Aggregate enforcement line counts

Print the total number of proxy denial lines and firewall-blocked egress lines in the exported invocation table.

In [ ]:
print(f"Total proxy_denied_count: {summary['proxy_denied_count'].sum()}")
print(f"Total blocked_egress_line_count: {summary['blocked_egress_line_count'].sum()}")

## Aggregate proxy access line count

Print the total number of proxy access log lines in the exported invocation table.

In [ ]:
print(f"Total proxy_access_line_count: {summary['proxy_access_line_count'].sum()}")

## Export derived CSV

The derived artifact keeps one row per Andvari invocation, but removes unrelated operational fields.

In [ ]:
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUTPUT_CSV, index=False)
OUTPUT_CSV


## Thesis summary table: agent by variant

This table corresponds to the compact Option B presentation.

In [ ]:
table = (
    summary
    .groupby(['agent', 'variant_label'], dropna=False)
    .agg(
        invocations=('run_id', 'count'),
        denied_blocked_invocations=('denied_or_blocked', 'sum'),
    )
    .reset_index()
)

table['share'] = table['denied_blocked_invocations'] / table['invocations']
table['share_percent'] = (table['share'] * 100).round(2)

agent_order = {'claude': 0, 'Claude': 0, 'codex': 1, 'Codex': 1}
variant_order = {
    'Best-effort': 0,
    'Slightly degraded': 1,
    'Moderately degraded': 2,
}

table['_agent_order'] = table['agent'].map(agent_order).fillna(99)
table['_variant_order'] = table['variant_label'].map(variant_order).fillna(99)
table = table.sort_values(['_agent_order', '_variant_order']).drop(columns=['_agent_order', '_variant_order'])

table


## Totals and validation checks

These checks make the expected denominator and event count explicit.

In [ ]:
total_invocations = len(summary)
total_denied_blocked = int(summary['denied_or_blocked'].sum())
proxy_denial_invocations = int((summary['proxy_denied_count'] > 0).sum())
firewall_block_invocations = int((summary['blocked_egress_line_count'] > 0).sum())

print(f'Total invocations: {total_invocations}')
print(f'Denied/blocked invocations: {total_denied_blocked}')
print(f'Invocations with proxy denial: {proxy_denial_invocations}')
print(f'Invocations with firewall block: {firewall_block_invocations}')

# Expected values for the current thesis dataset.
assert total_invocations == 240
assert total_denied_blocked == 13


## Generate LaTeX table

This cell prints a LaTeX table that can be pasted into the thesis.

In [ ]:
def latex_escape(value):
    return str(value).replace('_', r'\_')

lines = []
lines.append(r'\begin{table}[htbp]')
lines.append(r'\centering')
lines.append(r'\caption{Andvari invocations with denied proxy requests or firewall-blocked egress events by agent and reconstruction variant.}')
lines.append(r'\label{tab:proxy-firewall-events}')
lines.append(r'\begin{tabular}{llrrr}')
lines.append(r'\toprule')
lines.append(r'Agent & Variant & Invocations & Denied/blocked invocations & Share \\')
lines.append(r'\midrule')

previous_agent = None
for _, row in table.iterrows():
    agent = latex_escape(row['agent']).capitalize()
    variant = latex_escape(row['variant_label'])
    invocations = int(row['invocations'])
    denied = int(row['denied_blocked_invocations'])
    share = f"{row['share_percent']:.2f}\\%"

    if previous_agent is not None and agent != previous_agent:
        lines.append(r'\midrule')

    lines.append(f'{agent} & {variant} & {invocations} & {denied} & {share} \\\\')
    previous_agent = agent

total_share = total_denied_blocked / total_invocations * 100
lines.append(r'\midrule')
lines.append(f'Total & -- & {total_invocations} & {total_denied_blocked} & {total_share:.2f}\\% \\\\')
lines.append(r'\bottomrule')
lines.append(r'\end{tabular}')
lines.append(r'\end{table}')

latex_table = '\n'.join(lines)
print(latex_table)
